In [ ]:
import sys
from pathlib import Path
import datetime as dt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr, mannwhitneyu
from utils.py_eddy_tracker.observations.tracking import TrackEddiesObservations

EXPERIMENT = "gulf_stream_20241001_20250701"
COLORS = {"cyclone": "tab:blue", "anticyclone": "tab:red"}

PFT_COLS = ["Diatoms", "Dinoflagellates", "Haptophytes",
    "Cryptophytes", "Green_algae", "Cyanobacteria"]
FRAC_COLS = [f"{c}_frac" for c in PFT_COLS]

RADIAL_BINS = [0, 0.25, 0.5, 0.75, 1.0, 1.5]
bin_mids = [(RADIAL_BINS[i] + RADIAL_BINS[i + 1]) / 2
    for i in range(len(RADIAL_BINS) - 1)]
bin_labels = [f"{RADIAL_BINS[i]:.2f}-{RADIAL_BINS[i + 1]:.2f}"
    for i in range(len(RADIAL_BINS) - 1)]


def correct_holm_bonferroni(p_values):
    """
    Holm-Bonferroni step-down correction for multiple comparisons.

    Sorts p-values ascending, multiplies the k-th smallest by (m - k + 1), then enforces monotonicity so adjusted values never decrease. More powerful than plain Bonferroni because it penalizes later (less significant) tests less heavily.
    """
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m)
    adjusted[order] = p[order] * (m - np.arange(m))
    # enforce monotonicity: each adjusted p >= the previous
    adjusted[order] = np.maximum.accumulate(adjusted[order])
    return np.clip(adjusted, 0, 1)

In [ ]:
def compute_haversine_km(lon1, lat1, lon2, lat2):
    """Vectorized great-circle distance in km."""
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def load_track_props(polarity):
    """
    Load py-eddy-tracker zarr and return per-observation properties.

    Converts PET epoch (days since 1950-01-01) to calendar dates and computes lifetime as span of the track's time series.
    """
    PET_EPOCH = dt.date(1950, 1, 1)
    zarr_path = (ROOT / "outputs" / EXPERIMENT / "eddy_track"
        / polarity / f"{polarity}_tracks.zarr")
    tracked = TrackEddiesObservations.load_file(str(zarr_path))

    unique_ids = np.unique(tracked.track)
    lifetime = {}
    for tid in unique_ids:
        t = tracked.time[tracked.track == tid]
        lifetime[tid] = int(t.max() - t.min()) + 1

    dates = [PET_EPOCH + dt.timedelta(days=int(d)) for d in tracked.time]
    return pd.DataFrame({
        "track_id": tracked.track.astype(int),
        "date": pd.to_datetime(dates),
        "polarity": polarity,
        "center_lon": (tracked.longitude + 180) % 360 - 180,
        "center_lat": tracked.latitude,
        "radius_km": tracked.radius_e / 1000,
        "amplitude_m": tracked.amplitude,
        "speed_avg": tracked.speed_average,
        "lifetime_days": [lifetime[tid] for tid in tracked.track],
    })


def compute_radial_profile(df, col, n_boot=1000):
    """
    Bootstrap 95% CI for mean value per radial bin.

    Averages per eddy-date first to prevent pseudoreplication from large eddies having more pixels, then bootstraps over those means.
    """
    grouped = (df.groupby(["track_id", "date", "r_bin"], observed=True)[col]
        .mean().reset_index())

    means, lo, hi = [], [], []
    for label in bin_labels:
        vals = grouped.loc[grouped["r_bin"] == label, col].values
        if len(vals) < 3:
            means.append(np.nan); lo.append(np.nan); hi.append(np.nan)
            continue
        res = stats.bootstrap((vals,), np.mean, n_resamples=n_boot,
            confidence_level=0.95, method="percentile")
        means.append(vals.mean())
        lo.append(res.confidence_interval.low)
        hi.append(res.confidence_interval.high)
    return np.array(means), np.array(lo), np.array(hi)

## Data loading

In [ ]:
frames = []
for polarity in ("cyclone", "anticyclone"):
    pft_dir = ROOT / "outputs" / EXPERIMENT / "pft" / polarity
    for fp in sorted(pft_dir.glob("eddy_*_pfts.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        frames.append(df)

pfts = pd.concat(frames, ignore_index=True)
print(
    f"pixels: {len(pfts):,}\n"
    f"eddy_files: {len(frames)}"
)
pfts.head()

In [ ]:
track_properties = pd.concat(
    [load_track_props(p) for p in ("cyclone", "anticyclone")],
    ignore_index=True,
)

pfts = pfts.merge(
    track_properties[["track_id", "date", "polarity",
        "radius_km", "amplitude_m", "speed_avg", "lifetime_days"]],
    on=["track_id", "date", "polarity"],
    how="left",
)

# Normalized radial distance: how far each pixel is from eddy center, in units of the eddy's effective radius
pfts["dist_km"] = compute_haversine_km(
    pfts["pixel_lon"].values, pfts["pixel_lat"].values,
    pfts["center_lon"].values, pfts["center_lat"].values,
)
pfts["r_norm"] = pfts["dist_km"] / pfts["radius_km"]

print(
    f"merged_rows: {len(pfts):,}\n"
    f"missing_radius: {pfts['radius_km'].isna().sum()}"
)

In [ ]:
# PFT fractions: each PFT's Chla as a share of the total
# If fractions shift radially, community *composition* changes, not just total biomass
total_pft = pfts[PFT_COLS].sum(axis=1)

for col in PFT_COLS:
    pfts[f"{col}_frac"] = pfts[col] / total_pft

n_zero = (total_pft == 0).sum()
print(
    f"zero_total_pixels: {n_zero}\n"
    f"zero_total_percent: {n_zero / len(pfts) * 100:.2f}"
)

# Sanity check: fractions should sum to 1.0 per pixel
frac_sums = pfts[FRAC_COLS].sum(axis=1)
print(f"\nFraction row sums:\n{frac_sums.describe()}")

print("\nPer-PFT fraction summary:")
pfts[FRAC_COLS].describe().round(4)

## Summary statistics

In [ ]:
summary = (
    pfts.groupby("polarity")
    .apply(lambda g: pd.Series({
        "eddies": g["track_id"].nunique(),
        "eddy-dates": g.groupby("track_id")["date"].nunique().sum(),
        "pixels": len(g),
        "median lifetime (d)": g.drop_duplicates("track_id")["lifetime_days"].median(),
        "median radius (km)": g["radius_km"].median(),
        "median amplitude (m)": g["amplitude_m"].median(),
    }), include_groups=False)
)
summary

Probability density of per-eddy-date mean PFT concentrations for cyclonic (blue) and anticyclonic (red) eddies. Each observation is the spatial mean within one eddy on one day, using eddy-date means rather than raw pixels to avoid pseudoreplication from spatially correlated samples.

In [ ]:
eddy_means = (
    pfts.groupby(["track_id", "date", "polarity"])[PFT_COLS]
    .mean().reset_index()
)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(PFT_COLS):
    ax = axes.flat[i]
    for pol in ("cyclone", "anticyclone"):
        vals = eddy_means.loc[eddy_means["polarity"] == pol, col]
        ax.hist(vals, bins=30, alpha=0.5, label=pol,
            color=COLORS[pol], density=True)
    ax.set_title(col)
    ax.set_xlabel("mg/m\u00b3")
    if i == 0:
        ax.legend()

fig.suptitle("PFT concentration distributions (eddy-date means)", y=1.01)
fig.tight_layout()

## Radial profiles - absolute PFT concentrations

In [ ]:
pfts["r_bin"] = pd.cut(pfts["r_norm"], bins=RADIAL_BINS,
    labels=bin_labels, right=False)
pft_binned = pfts.dropna(subset=["r_bin"])
print(
    f"pixels_within_1_5_r: {len(pft_binned):,}\n"
    f"percent_of_total: {len(pft_binned) / len(pfts) * 100:.1f}"
)

Mean PFT concentration (mg/m³) as a function of normalized radial distance (r/R), with 95% bootstrap confidence intervals. Each radial bin averages per-eddy-date bin means to weight eddies equally. Separation between profiles indicates polarity-dependent concentration gradients.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
x = np.array(bin_mids)

for i, col in enumerate(PFT_COLS):
    ax = axes.flat[i]
    for pol in ("cyclone", "anticyclone"):
        subset = pft_binned[pft_binned["polarity"] == pol]
        m, lo, hi = compute_radial_profile(subset, col)
        ax.plot(x, m, "o-", color=COLORS[pol], label=pol, ms=4)
        ax.fill_between(x, lo, hi, color=COLORS[pol], alpha=0.15)
    ax.set_title(col)
    ax.set_xlabel("r / R")
    ax.set_ylabel("mg/m\u00b3")
    ax.set_xlim(0, 1.5)
    if i == 0:
        ax.legend()

fig.suptitle("Radial PFT concentration profiles", y=1.01)
fig.tight_layout()

## Radial profiles - PFT fractions

If concentration profiles shift but fraction profiles stay flat, the gradient
is a total-biomass effect (everything scales together). If fractions shift too,
community composition genuinely changes with radial distance - a stronger
ecological claim.

Radial profiles of PFT fraction (each PFT's Chla as a share of total Chla) with 95% bootstrap CIs. If absolute concentration profiles diverge but fraction profiles remain flat, the gradient is a total-biomass effect rather than a community composition shift.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
x = np.array(bin_mids)

for i, col in enumerate(FRAC_COLS):
    ax = axes.flat[i]
    display_name = col.replace("_frac", "")
    for pol in ("cyclone", "anticyclone"):
        subset = pft_binned[pft_binned["polarity"] == pol]
        m, lo, hi = compute_radial_profile(subset, col)
        ax.plot(x, m, "o-", color=COLORS[pol], label=pol, ms=4)
        ax.fill_between(x, lo, hi, color=COLORS[pol], alpha=0.15)
    ax.set_title(display_name)
    ax.set_xlabel("r / R")
    ax.set_ylabel("fraction of total Chla")
    ax.set_xlim(0, 1.5)
    if i == 0:
        ax.legend()

fig.suptitle("Radial PFT fraction profiles - does community composition shift?",
    y=1.01)
fig.tight_layout()

Cyclone-minus-anticyclone difference in PFT fraction at each radial bin, with 95% bootstrap confidence intervals (1000 resamples). Where the CI excludes zero, the composition difference is statistically significant at that normalized distance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
x = np.array(bin_mids)

for i, col in enumerate(FRAC_COLS):
    ax = axes.flat[i]
    display_name = col.replace("_frac", "")

    grouped = (pft_binned.groupby(["track_id", "date", "r_bin", "polarity"],
        observed=True)[col].mean().reset_index())

    diffs, lo, hi = [], [], []
    for label in bin_labels:
        cyc_vals = grouped.loc[(grouped["r_bin"] == label) &
            (grouped["polarity"] == "cyclone"), col].values
        anti_vals = grouped.loc[(grouped["r_bin"] == label) &
            (grouped["polarity"] == "anticyclone"), col].values
        if len(cyc_vals) < 3 or len(anti_vals) < 3:
            diffs.append(np.nan); lo.append(np.nan); hi.append(np.nan)
            continue

        rng = np.random.default_rng(42)
        boot_diffs = []
        for _ in range(1000):
            c = rng.choice(cyc_vals, size=len(cyc_vals), replace=True)
            a = rng.choice(anti_vals, size=len(anti_vals), replace=True)
            boot_diffs.append(c.mean() - a.mean())
        boot_diffs = np.array(boot_diffs)
        diffs.append(cyc_vals.mean() - anti_vals.mean())
        lo.append(np.percentile(boot_diffs, 2.5))
        hi.append(np.percentile(boot_diffs, 97.5))

    diffs, lo, hi = np.array(diffs), np.array(lo), np.array(hi)
    # Significant if CI excludes zero at all bins
    all_above = np.all(lo[~np.isnan(lo)] > 0)
    all_below = np.all(hi[~np.isnan(hi)] < 0)
    if all_above:
        shade_color = "tab:blue"
    elif all_below:
        shade_color = "tab:red"
    else:
        shade_color = "gray"
    ax.plot(x, diffs, "o-", color=shade_color, ms=4)
    ax.fill_between(x, lo, hi, color=shade_color, alpha=0.15)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(display_name)
    ax.set_xlabel("r / R")
    ax.set_ylabel("\u0394 fraction (cyc \u2212 anti)")
    ax.set_xlim(0, 1.5)

fig.suptitle("Cyclone \u2212 anticyclone fraction difference by radial bin",
    y=1.01)
fig.tight_layout()